# Deep Learning Facial Image Classification

This notebook is a concise walkthrough of the final two-stage ResNet50 pipeline. Reusable implementation lives in `src/data.py` and `src/train_resnet.py`.

> This project is an image-classification exercise, not a screening or diagnostic system. Facial appearance cannot establish an autism diagnosis.


## Recorded final run

The final run used a fixed-seed, stratified image-level 80/10/10 split of 2,526 images. The fine-tuned ResNet50 reached 87.9% validation accuracy, then 83.5% accuracy and 0.917 ROC-AUC on the held-out test set.

If the dataset contains multiple images of one person, replace the image-level split with an identity-disjoint split before treating the evaluation as reliable.


## Setup

Run from the repository root after installing `requirements.txt`. Set `DATASET_DIR` to a directory containing `autistic/` and `non_autistic/` subdirectories.


In [ ]:
from pathlib import Path

from src.data import discover_labeled_images, stratified_split
from src.train_resnet import TrainingConfig, train


In [ ]:
DATASET_DIR = Path("path/to/dataset")
OUTPUT_DIR = Path("output/resnet50")
CONFIG = TrainingConfig(seed=42)


## Verify the split

The split is stratified and deterministic. Test paths remain untouched until the selected checkpoint is evaluated.


In [ ]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Update DATASET_DIR before running this cell: {DATASET_DIR}"
    )

paths, labels = discover_labeled_images(
    DATASET_DIR,
    ("autistic", "non_autistic"),
)
split = stratified_split(paths, labels, seed=CONFIG.seed)
{
    "train": len(split.train_paths),
    "validation": len(split.validation_paths),
    "test": len(split.test_paths),
}


## Train, select, and evaluate

Stage 1 trains the classifier head with the ImageNet base frozen. Stage 2 fine-tunes the final 20 ResNet layers at a lower learning rate while keeping batch-normalization layers frozen. The better checkpoint is selected by validation loss and evaluated on the test set once.


In [ ]:
metrics = train(DATASET_DIR, OUTPUT_DIR, CONFIG)
metrics
